In [1]:
# Cell 1 — which machine did I get?
!nvidia-smi
import timm
import torch

print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(), "| timm", timm.__version__)

Wed Sep 16 09:38:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — fetch and install my code
%cd /kaggle/working
!git clone https://github.com/stavhadas/ai-cv-training-project-1.git pcb-inspector
%cd pcb-inspector
!pip install -e . -q
!pcbi --help

/kaggle/working
Cloning into 'pcb-inspector'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 32 (delta 0), reused 29 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 17.65 KiB | 4.41 MiB/s, done.
/kaggle/working/pcb-inspector
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pcbi (pyproject.toml) ... done
                                                                                
 Usage: pcbi [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
 PCB solder-joint inspector.                                                    
                       

In [3]:
# Cell 3 — load the W&B key; fall back to offline mode if missing
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    os.environ["WANDB_MODE"] = "offline"
    print("No W&B key found; logging offline")

In [4]:
# Cell 4 — where is the dataset?
!ls -R /kaggle/input | head -50

/kaggle/input:
datasets

/kaggle/input/datasets:
mauriziocalabrese

/kaggle/input/datasets/mauriziocalabrese:
soldef-ai-pcb-dataset-for-defect-detection

/kaggle/input/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection:
SolDef_AI

/kaggle/input/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection/SolDef_AI:
Dataset
Labeled

/kaggle/input/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection/SolDef_AI/Dataset:
CS1
CS2
CS3
CS4
CS5
CS6
CS7

/kaggle/input/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection/SolDef_AI/Dataset/CS1:
R0805
R1206

/kaggle/input/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection/SolDef_AI/Dataset/CS1/R0805:
V1
V2
V2.1
V3

/kaggle/input/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection/SolDef_AI/Dataset/CS1/R0805/V1:
Setup1
Setup3

/kaggle/input/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection/SolDef_AI/Dataset/CS1/R0805/V1/Setup1:
WIN_

In [ ]:
# Cell 5 — mount the crops and prove they are the ones this run thinks they are
#
# The crops come from `pcbi publish-crops` as a private dataset; attach it under Data in the
# sidebar. Two things can go wrong quietly, so both are checked every session rather than once
# by eye:
#
#   1. `--dir-mode zip` uploads crops/ as an archive. Kaggle normally expands it, but if it has
#      not, the fallback below unpacks it once into working storage and the run continues.
#   2. The attached dataset may be an older version than the split this run assumes. The split
#      hash is the only thing that catches that, which is the whole reason it exists.
import csv
import json
import zipfile
from pathlib import Path

EXPECTED_CROPS = 400

attached = sorted(Path("/kaggle/input").glob("*/manifest_meta.json"))
assert attached, "crops dataset not attached — add it under Data in the notebook sidebar"
DATA = attached[0].parent
print("dataset:", DATA)

crops = DATA / "crops"
if not crops.is_dir():
    archive = next(iter(DATA.glob("crops*.zip")), None)
    assert archive, f"no crops/ and no crops zip in {DATA}: {[p.name for p in DATA.iterdir()]}"
    unpacked = Path("/kaggle/working/crops_unpacked")
    if not unpacked.is_dir():
        with zipfile.ZipFile(archive) as archive_file:
            archive_file.extractall(unpacked)
    found = sorted(unpacked.rglob("*.png"))  # the zip may or may not carry a top-level crops/
    assert found, f"{archive.name} held no PNGs"
    crops = found[0].parent
    print(f"Kaggle left {archive.name} packed; expanded it to {crops}")

pngs = sorted(crops.glob("*.png"))
assert len(pngs) == EXPECTED_CROPS, f"expected {EXPECTED_CROPS} crops, found {len(pngs)}"

split_hash = json.loads((DATA / "manifest_meta.json").read_text())["split_hash"]
with (DATA / "split_v1.csv").open(newline="") as handle:
    recorded = {row["split_hash"] for row in csv.DictReader(handle)}
assert recorded == {split_hash}, f"split_v1.csv disagrees with manifest_meta.json: {recorded}"

with (DATA / "manifest.csv").open(newline="") as handle:
    manifest = list(csv.DictReader(handle))
assert len(manifest) == EXPECTED_CROPS, f"manifest has {len(manifest)} rows, not {EXPECTED_CROPS}"
assert {row["crop_id"] for row in manifest} == {p.stem for p in pngs}, "manifest/PNG mismatch"

print(f"{len(pngs)} crops · {len(manifest)} manifest rows · crops at {crops}")
print(f"split hash: {split_hash}")
print("Log that hash with every run. A run reporting a different one trained on different data.")